In [26]:
from torchvision.datasets import CIFAR10
from torch.utils.data import Dataset
from torchvision import transforms

transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])

train_base = CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=transform
)

test_base = CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=transform
)

class CatsDogsDataset(Dataset):
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

        # CIFAR-10: cat = 3, dog = 5
        self.indices = [
            i for i, label in enumerate(base_dataset.targets)
            if label in [3, 5]
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, original_label = self.base_dataset[
            self.indices[idx]
        ]

        # Remap labels: cat = 0, dog = 1
        label = 0 if original_label == 3 else 1

        return image, label


trainset = CatsDogsDataset(train_base)
testset = CatsDogsDataset(test_base)

print("Training images:", len(trainset))
print("Testing images:", len(testset))
print("Image shape:", trainset[0][0].shape)
print("Cat label:", trainset[0][1])

Training images: 10000
Testing images: 2000
Image shape: torch.Size([3, 32, 32])
Cat label: 0


In [27]:
from torch.utils.data import DataLoader

trainloader = DataLoader(
    trainset,
    batch_size=64,
    shuffle=True,
    num_workers=0
)

testloader = DataLoader(
    testset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)

images, labels = next(iter(trainloader))

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Unique labels in batch:", labels.unique())

Images: torch.Size([64, 3, 32, 32])
Labels: torch.Size([64])
Unique labels in batch: tensor([0, 1])


In [28]:
import torch
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 2)  # Only Cat and Dog
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

In [29]:
import torch.optim as optim
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),
            lr=0.001,
            weight_decay=1e-4
)

print("Model recreated successfully")
print("Device:", device)

Model recreated successfully
Device: cpu


In [ ]:
epochs = 10

train_losses = []
train_accuracies = []

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

Epoch 1/10 | Loss: 0.6405 | Accuracy: 62.22%
Epoch 2/10 | Loss: 0.5586 | Accuracy: 71.57%


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.numpy())
        all_predictions.extend(predictions.cpu().numpy())

print(
    "Test Accuracy:",
    round(accuracy_score(all_labels, all_predictions) * 100, 2),
    "%"
)

print(
    "Precision:",
    round(precision_score(all_labels, all_predictions), 4)
)

print(
    "Recall:",
    round(recall_score(all_labels, all_predictions), 4)
)

print(
    "F1-score:",
    round(f1_score(all_labels, all_predictions), 4)
)

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1],
    target_names=["Cat", "Dog"]
))

In [ ]:
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())

print("Total actual labels:", len(all_labels))
print("Total predictions:", len(all_predictions))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(all_labels, all_predictions)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Cat", "Dog"]
 ).plot(cmap="Blues")

plt.title("Cats vs Dogs Classification")
plt.show()

In [ ]:
import sys
print(sys.executable)

import matplotlib
print(matplotlib.__version__)
print(matplotlib.__file__)